# Loans at Risk: Capturing Default — Extract · Transform · Load

## Purpose

This notebook implements the ETL layer of the LendingClub default prediction project.  
The training window covers **2007–2014**; the test window covers **2015**.

The objective is not analysis. It is structural control. Raw CSV extracts are converted into datasets that are suitable for downstream modeling under a clearly defined constraint: prediction is made at the moment of application submission.“Structurally reliable” in this context means:

- Identical schema across train and test  
- Explicit treatment of datatypes and null structure  
- Removal of null-only and constant columns  
- Elimination of export artifacts  
- Early enforcement of the submission-time prediction boundary  

No interpretation is performed here. This notebook establishes a clean, governed input layer so that later analytical steps operate on stable ground.
All dependency management is handled at repository level (`README`, `requirements.txt`). This notebook assumes the environment is already configured.

---

## Extract

### Raw Source Normalization

The archived LendingClub CSV releases contain several non-loan rows that are
included as part of the original export format. These rows are not loan-level
records and must be removed before the files can be treated as tabular data.

Typical examples include:

- Prospectus or informational text rows at the top of the file  
- Repeated header rows embedded within the dataset  
- Footer summary rows such as  
  *“Total amount funded in policy code 1”* and  
  *“Total amount funded in policy code 2”*

These rows are artifacts of the original export format rather than actual
observations. They are removed during raw ingestion before any schema
validation or transformation logic is applied.

After normalization, the source files are concatenated into two
canonical raw datasets:

- **2007–2014** → training dataset  
- **2015** → testing dataset  

The resulting files contain loan-level records only and serve as the raw input
layer for the ETL pipeline. 

Extraction preserves the raw state of the data (after source normalization). No filtering, type coercion, or feature selection occurs at this stage. The raw layer is treated as immutable input.

---

## Transform

The transformation phase enforces structural and temporal discipline.

Concretely:

- Schema and datatype validation  
- Identification and removal of null-only and constant columns  
- Detection of mixed types and object-encoded numerics  
- Removal of structural artifacts (e.g., exported index columns)  
- Explicit exclusion of post-origination, underwriting, and pricing variables  
- Deterministic type conversions (including numeric and date normalization)  
- Structured handling of credit-event timing variables, where null represents absence rather than data loss  
- Identical transformation logic across training and test sets  

No feature engineering beyond structural normalization is performed.  
No modeling assumptions are introduced.

The output of this phase is a submission-time compliant dataset.

---

## Load

The transformed datasets are persisted in Parquet format. Two layers are materialized:

- **Clean dataset** → structurally aligned variables  
- **Feature-base dataset** → submission-time eligible variables plus target  

Schema identity between training and test is validated prior to persistence.
These outputs form the controlled input layer for EDA and modeling.

---

## Scope Boundary

Feature engineering, modeling, evaluation, and decision logic are handled in subsequent notebooks.
This notebook exists to ensure that those stages operate on data that is structurally sound, temporally consistent, and reproducible.

---

In [1]:
from pathlib import Path
import sys

current_path = Path.cwd().resolve()
project_root = None

for parent_path in (current_path, *current_path.parents):
    if (parent_path / "pyproject.toml").exists():
        project_root = parent_path
        break

if project_root is None:
    raise RuntimeError(
        f"Failed to resolve project root: pyproject.toml not found from {current_path}"
    )

src_path = project_root / "src"
if not src_path.exists():
    raise RuntimeError(f"Expected 'src' directory at: {src_path}")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

{
    "stage": "bootstrap_import_path_ready",
    "project_root": str(project_root),
}

{'stage': 'bootstrap_import_path_ready',
 'project_root': 'D:\\Portfolio\\loans-at-risk-capturing-default'}

In [2]:

# ============================================================================
# Imports: libraries and custom functions
# ============================================================================

from datetime import datetime, timezone
from typing import Callable
import uuid
from IPython.display import Markdown, display
import csv

import pandas as pd

# Editable-installed local package
import config.logging as log_config
import dataset.preprocessing as pp
import analysis.etl_artifacts as etla
import analysis.dataset_artifacts as da
import features.feature_utils as fu

In [3]:
# ===============================
# Paths and run context
# ===============================

logs_root = project_root / "logs"
logs_root.mkdir(parents=True, exist_ok=True)

PROJECT_LOG_FILE = logs_root / "project.log"

run_id = uuid.uuid4().hex[:10]
run_timestamp_utc = datetime.now(timezone.utc)

run_header = (
    "NEW RUN: "
    f"{run_timestamp_utc.strftime('%Y-%m-%d %H:%M:%S')} UTC | "
    f"RUN_ID={run_id}"
)

log_config.log_messages("\n" + "=" * 60, PROJECT_LOG_FILE)
log_config.log_messages(run_header, PROJECT_LOG_FILE)
log_config.log_messages("=" * 60 + "\n", PROJECT_LOG_FILE)

log: Callable[[str], None] = log_config.get_logger(PROJECT_LOG_FILE)
log("ETL notebook initialized")
log(f"project_root: {project_root}")

run_metadata = {
    "stage": "run_started",
    "run_id": run_id,
    "utc_timestamp": run_timestamp_utc.isoformat(),
}


# ---------------------------------------------------------------
# Inputs for this notebook (raw, interim, report)
# ---------------------------------------------------------------

# External source datasets
external_2007_2011_data_file = project_root / "data" / "external" / "loan_data_2007-2011.csv"
external_2012_2013_data_file = project_root / "data" / "external" / "loan_data_2012-2013.csv"
external_2014_data_file = project_root / "data" / "external" / "loan_data_2014.csv"
external_2015_data_file = project_root / "data" / "external" / "loan_data_2015.csv"

# Staged normalized source datasets
staged_2007_2011_data_file = project_root / "data" / "staging" / "normalized_loan_data_2007_2011.csv"
staged_2012_2013_data_file = project_root / "data" / "staging" / "normalized_loan_data_2012_2013.csv"
staged_2014_data_file = project_root / "data" / "staging" / "normalized_loan_data_2014.csv"
staged_2015_data_file = project_root / "data" / "staging" / "normalized_loan_data_2015.csv"

# Raw datasets
raw_train_data_file = project_root / "data" / "raw" / "loan_data_2007_2014.csv"
raw_test_data_file = project_root / "data" / "raw" / "loan_data_2015.csv"

# Clean datasets
clean_train_data_file = project_root / "data" / "interim" / "clean_loan_data_2007_2014.parquet"
clean_test_data_file = project_root / "data" / "interim" / "clean_loan_data_2015.parquet"

# Feature base datasets
feature_base_train_data_file = project_root / "data" / "interim" / "feature_base_loan_data_2007_2014.parquet"
feature_base_test_data_file = project_root / "data" / "interim" / "feature_base_loan_data_2015.parquet"

# Audit output paths
audit_dir = project_root / "artifacts" / "audit"

appendix_base_name = "appendix_A_raw_feature_universe"

appendix_parquet_file = audit_dir / f"{appendix_base_name}.parquet"
appendix_csv_file = audit_dir / f"{appendix_base_name}.csv"

required_directories = {
	staged_2007_2011_data_file.parent,
	staged_2012_2013_data_file.parent,
	staged_2014_data_file.parent,
	staged_2015_data_file.parent,
    external_2007_2011_data_file.parent,
	external_2012_2013_data_file.parent,
	external_2014_data_file.parent,
	external_2015_data_file.parent,
    raw_train_data_file.parent,
    raw_test_data_file.parent,
    clean_train_data_file.parent,
    clean_test_data_file.parent,
    feature_base_train_data_file.parent,
    feature_base_test_data_file.parent,
    appendix_parquet_file.parent,
    appendix_csv_file.parent,
}

for directory_path in sorted(required_directories, key=lambda path: str(path)):
    directory_path.mkdir(parents=True, exist_ok=True)

log(f"External 2007-2011 data path: {external_2007_2011_data_file}")
log(f"External 2012-2013 data path: {external_2012_2013_data_file}")
log(f"External 2014 data path: {external_2014_data_file}")
log(f"External 2015 data path: {external_2015_data_file}")
log(f"Staged 2007-2011 data path: {staged_2007_2011_data_file}")
log(f"Staged 2012-2013 data path: {staged_2012_2013_data_file}")
log(f"Staged 2014 data path: {staged_2014_data_file}")
log(f"Staged 2015 data path: {staged_2015_data_file}")
log(f"Raw training data path: {raw_train_data_file}")
log(f"Raw test data path: {raw_test_data_file}")
log(f"Clean training parquet path: {clean_train_data_file}")
log(f"Clean test parquet path: {clean_test_data_file}")
log(f"Feature base training parquet path: {feature_base_train_data_file}")
log(f"Feature base test parquet path: {feature_base_test_data_file}")
log(f"Appendix parquet path: {appendix_parquet_file}")
log(f"Appendix CSV path: {appendix_csv_file}")

run_metadata

{'stage': 'run_started',
 'run_id': '680153ab71',
 'utc_timestamp': '2026-04-03T23:19:34.194295+00:00'}

## Raw Source Normalization

Before we can use the dataset we need to make sure that we actually deal with structural data and that all files share the same schemas. 

In [4]:
# ---------------------------------------------------------------
# Inspect source file structure before parsing (head + tail)
# ---------------------------------------------------------------

log("Starting source structure inspection")

source_file_paths = [
	external_2007_2011_data_file,
	external_2012_2013_data_file,
	external_2014_data_file,
	external_2015_data_file,
]

preview_line_count = 8
preview_character_limit = 250

for source_file_path in source_file_paths:

	log(f"Inspecting source structure: {source_file_path.name}")

	try:

		with open(source_file_path, "r", encoding="utf-8", errors="replace") as file_handle:
			all_lines = file_handle.readlines()

		head_lines = all_lines[:preview_line_count]
		tail_lines = all_lines[-preview_line_count:]

		preview_records: list[dict[str, str | int]] = []

		for line_number, line in enumerate(head_lines):
			preview_records.append(
				{
					"section": "head",
					"line_number": line_number,
					"preview": line.rstrip("\n")[:preview_character_limit],
				}
			)

		for line_offset, line in enumerate(tail_lines):
			preview_records.append(
				{
					"section": "tail",
					"line_number": len(all_lines) - preview_line_count + line_offset,
					"preview": line.rstrip("\n")[:preview_character_limit],
				}
			)

	except Exception as exc:

		log(f"Source structure inspection failed for {source_file_path.name} | error={exc}")
		raise

	df_source_structure_preview = pd.DataFrame(preview_records)

	display(Markdown(f"### Source structure preview: `{source_file_path.name}`"))
	display(df_source_structure_preview)

	log(f"Completed source structure inspection: {source_file_path.name}")

### Source structure preview: `loan_data_2007-2011.csv`

,section,line_number,preview
0,head,0,Notes offered by Prospectus (https://www.lendi...
1,head,1,"""id"",""member_id"",""loan_amnt"",""funded_amnt"",""fu..."
2,head,2,"""1077501"",""1296599"",""5000"",""5000"",""4975"","" 36 ..."
3,head,3,"""1077430"",""1314167"",""2500"",""2500"",""2500"","" 60 ..."
4,head,4,"""1077175"",""1313524"",""2400"",""2400"",""2400"","" 36 ..."
5,head,5,"""1076863"",""1277178"",""10000"",""10000"",""10000"","" ..."
6,head,6,"""1075358"",""1311748"",""3000"",""3000"",""3000"","" 60 ..."
7,head,7,"""1075269"",""1311441"",""5000"",""5000"",""5000"","" 36 ..."
8,tail,42536,"""72998"",""72992"",""1000"",""1000"",""0"","" 36 months""..."
9,tail,42537,"""72176"",""70868"",""2525"",""2525"",""225"","" 36 month..."


### Source structure preview: `loan_data_2012-2013.csv`

,section,line_number,preview
0,head,0,Notes offered by Prospectus (https://www.lendi...
1,head,1,"""id"",""member_id"",""loan_amnt"",""funded_amnt"",""fu..."
2,head,2,"""10129403"",""11981032"",""7550"",""7550"",""7550"","" 3..."
3,head,3,"""10149342"",""12000897"",""27050"",""27050"",""27050"",..."
4,head,4,"""10159584"",""12011200"",""9750"",""9750"",""9750"","" 3..."
5,head,5,"""10159498"",""1319523"",""12000"",""12000"",""12000"",""..."
6,head,6,"""10129506"",""11981122"",""20800"",""20800"",""20800"",..."
7,head,7,"""10159548"",""12011167"",""15000"",""15000"",""15000"",..."
8,tail,188121,"""1059394"",""1291010"",""15000"",""15000"",""15000"","" ..."
9,tail,188122,"""1059224"",""1290827"",""35000"",""35000"",""35000"","" ..."


### Source structure preview: `loan_data_2014.csv`

,section,line_number,preview
0,head,0,Notes offered by Prospectus (https://www.lendi...
1,head,1,"""id"",""member_id"",""loan_amnt"",""funded_amnt"",""fu..."
2,head,2,"""38098114"",""40860827"",""15000"",""15000"",""15000"",..."
3,head,3,"""36805548"",""39558264"",""10400"",""10400"",""10400"",..."
4,head,4,"""37612354"",""40375473"",""12800"",""12800"",""12800"",..."
5,head,5,"""37662224"",""40425321"",""7650"",""7650"",""7650"","" 3..."
6,head,6,"""37822187"",""40585251"",""9600"",""9600"",""9600"","" 3..."
7,head,7,"""37842129"",""40605224"",""21425"",""21425"",""21425"",..."
8,tail,235627,"""9684700"",""11536848"",""22000"",""22000"",""22000"",""..."
9,tail,235628,"""9584776"",""11436914"",""20700"",""20700"",""20700"",""..."


### Source structure preview: `loan_data_2015.csv`

,section,line_number,preview
0,head,0,Notes offered by Prospectus (https://www.lendi...
1,head,1,"""id"",""member_id"",""loan_amnt"",""funded_amnt"",""fu..."
2,head,2,"""68377006"",""73266831"",""6000"",""6000"",""6000"","" 3..."
3,head,3,"""68407301"",""73297138"",""27500"",""27500"",""27500"",..."
4,head,4,"""68527009"",""73416854"",""20000"",""20000"",""20000"",..."
5,head,5,"""68587652"",""73477494"",""25000"",""25000"",""25000"",..."
6,head,6,"""68446820"",""73336644"",""10000"",""10000"",""10000"",..."
7,head,7,"""68596180"",""73485987"",""20000"",""20000"",""20000"",..."
8,tail,421093,"""36441262"",""39152692"",""24000"",""24000"",""24000"",..."
9,tail,421094,"""36271333"",""38982739"",""13000"",""13000"",""13000"",..."


In [5]:
# ---------------------------------------------------------------
# Check that all source files have the same header structure
# ---------------------------------------------------------------

log("Starting schema consistency validation")

header_sets = {}

for source_file_path in source_file_paths:

	log(f"Reading schema header: {source_file_path.name}")

	try:

		with open(source_file_path, "r", encoding="utf-8", errors="replace") as file_handle:
			file_handle.readline()  # skip prospectus row
			header_line = file_handle.readline()

	except Exception as exc:

		log(f"Schema validation failed for {source_file_path.name} | error={exc}")
		raise

	column_names = tuple(col.strip('"') for col in header_line.strip().split(","))

	header_sets[source_file_path.name] = column_names


reference_schema = next(iter(header_sets.values()))

schema_comparison = [
	{
		"source_file": file_name,
		"matches_reference_schema": column_names == reference_schema
	}
	for file_name, column_names in header_sets.items()
]

df_schema_comparison = pd.DataFrame(schema_comparison)

display(df_schema_comparison)

log("Schema consistency validation completed")

,source_file,matches_reference_schema
0,loan_data_2007-2011.csv,True
1,loan_data_2012-2013.csv,True
2,loan_data_2014.csv,True
3,loan_data_2015.csv,True


Source inspection confirms that all LendingClub export files share the same
non-tabular artifacts and the same header schema. Normalization can therefore
be applied consistently across all source files before concatenation.

In [6]:
# ---------------------------------------------------------------
# Normalize external source files into staging
# ---------------------------------------------------------------

external_to_staging_file_paths = [
	(external_2007_2011_data_file, staged_2007_2011_data_file),
	(external_2012_2013_data_file, staged_2012_2013_data_file),
	(external_2014_data_file, staged_2014_data_file),
	(external_2015_data_file, staged_2015_data_file),
]

staging_normalization_records: list[dict[str, str | int]] = []

for external_file_path, staged_file_path in external_to_staging_file_paths:

	log(f"Starting normalization: {external_file_path.name}")

	source_row_count = 0
	staged_row_count = 0
	removed_non_tabular_row_count = 0
	removed_repeated_header_row_count = 0
	removed_blank_row_count = 0

	try:
		with open(external_file_path, "r", encoding="utf-8", errors="replace", newline="") as source_file_handle, \
			 open(staged_file_path, "w", encoding="utf-8", newline="") as staged_file_handle:

			csv_reader = csv.reader(source_file_handle)
			csv_writer = csv.writer(staged_file_handle)

			header_row: list[str] | None = None
			expected_column_count: int | None = None

			for row in csv_reader:
				source_row_count += 1

				if not row or all(cell.strip() == "" for cell in row):
					removed_blank_row_count += 1
					continue

				if header_row is None:
					if len(row) == 1:
						removed_non_tabular_row_count += 1
						continue

					header_row = row
					expected_column_count = len(header_row)
					csv_writer.writerow(header_row)
					staged_row_count += 1
					continue

				if row == header_row:
					removed_repeated_header_row_count += 1
					continue

				if len(row) != expected_column_count:
					removed_non_tabular_row_count += 1
					continue

				csv_writer.writerow(row)
				staged_row_count += 1

	except Exception as exc:
		log(
			f"Normalization failed for {external_file_path.name} → {staged_file_path.name} | "
			f"error={exc}"
		)
		raise

	log(
		f"Normalization completed: {external_file_path.name} → {staged_file_path.name} | "
		f"source_rows={source_row_count}, staged_rows={staged_row_count}, "
		f"removed_non_tabular={removed_non_tabular_row_count}, "
		f"removed_repeated_header={removed_repeated_header_row_count}, "
		f"removed_blank={removed_blank_row_count}"
	)

	staging_normalization_records.append(
		{
			"source_file": external_file_path.name,
			"staged_file": staged_file_path.name,
			"source_row_count": source_row_count,
			"staged_row_count": staged_row_count,
			"removed_non_tabular_row_count": removed_non_tabular_row_count,
			"removed_repeated_header_row_count": removed_repeated_header_row_count,
			"removed_blank_row_count": removed_blank_row_count,
		}
	)

df_staging_normalization_summary = pd.DataFrame(staging_normalization_records)

df_staging_normalization_summary

,source_file,staged_file,source_row_count,staged_row_count,removed_non_tabular_row_count,removed_repeated_header_row_count,removed_blank_row_count
0,loan_data_2007-2011.csv,normalized_loan_data_2007_2011.csv,42544,42536,4,0,4
1,loan_data_2012-2013.csv,normalized_loan_data_2012_2013.csv,188129,188124,3,0,2
2,loan_data_2014.csv,normalized_loan_data_2014.csv,235635,235630,3,0,2
3,loan_data_2015.csv,normalized_loan_data_2015.csv,421101,421096,3,0,2


In [7]:
df_staged_2007_2011_data = pd.read_csv(staged_2007_2011_data_file, low_memory=False)
df_staged_2012_2013_data = pd.read_csv(staged_2012_2013_data_file, low_memory=False)
df_staged_2014_data = pd.read_csv(staged_2014_data_file, low_memory=False)
df_staged_2015_data = pd.read_csv(staged_2015_data_file, low_memory=False)

pd.DataFrame(
	[
		{
			"source_file": "loan_data_2007-2011.csv",
			"row_count": df_staged_2007_2011_data.shape[0],
			"column_count": df_staged_2007_2011_data.shape[1],
			"unique_id_count": df_staged_2007_2011_data["id"].nunique(),
		},
		{
			"source_file": "loan_data_2012-2013.csv",
			"row_count": df_staged_2012_2013_data.shape[0],
			"column_count": df_staged_2012_2013_data.shape[1],
			"unique_id_count": df_staged_2012_2013_data["id"].nunique(),
		},
		{
			"source_file": "loan_data_2014.csv",
			"row_count": df_staged_2014_data.shape[0],
			"column_count": df_staged_2014_data.shape[1],
			"unique_id_count": df_staged_2014_data["id"].nunique(),
		},
		{
			"source_file": "loan_data_2015.csv",
			"row_count": df_staged_2015_data.shape[0],
			"column_count": df_staged_2015_data.shape[1],
			"unique_id_count": df_staged_2015_data["id"].nunique(),
		},
	]
)

,source_file,row_count,column_count,unique_id_count
0,loan_data_2007-2011.csv,42535,111,42535
1,loan_data_2012-2013.csv,188123,111,188123
2,loan_data_2014.csv,235629,111,235629
3,loan_data_2015.csv,421095,111,421095


In [8]:
# ---------------------------------------------------------------
# Concatenate staged datasets into canonical raw datasets
# ---------------------------------------------------------------

log("Starting staged dataset concatenation into raw datasets")

try:

	log("Loading staged source files")

	df_staged_2007_2011_data = pd.read_csv(staged_2007_2011_data_file, low_memory=False)
	df_staged_2012_2013_data = pd.read_csv(staged_2012_2013_data_file, low_memory=False)
	df_staged_2014_data = pd.read_csv(staged_2014_data_file, low_memory=False)
	df_staged_2015_data = pd.read_csv(staged_2015_data_file, low_memory=False)

	log("Concatenating staged training sources (2007–2014)")

	df_raw_2007_2014_data = pd.concat(
		[
			df_staged_2007_2011_data,
			df_staged_2012_2013_data,
			df_staged_2014_data,
		],
		axis=0,
		ignore_index=True,
	)

	log("Saving raw datasets")

	df_raw_2007_2014_data.to_csv(raw_train_data_file, index=False)
	df_staged_2015_data.to_csv(raw_test_data_file, index=False)

	log(
		f"Raw dataset written: {raw_train_data_file.name} | "
		f"rows={df_raw_2007_2014_data.shape[0]}, columns={df_raw_2007_2014_data.shape[1]}"
	)

	log(
		f"Raw dataset written: {raw_test_data_file.name} | "
		f"rows={df_staged_2015_data.shape[0]}, columns={df_staged_2015_data.shape[1]}"
	)

except Exception as exc:

	log(f"Raw dataset creation failed | error={exc}")
	raise

log("Raw dataset creation completed")

df_raw_dataset_creation_summary = pd.DataFrame(
	[
		{
			"raw_dataset_file": raw_train_data_file.name,
			"row_count": df_raw_2007_2014_data.shape[0],
			"column_count": df_raw_2007_2014_data.shape[1],
			"file_exists": raw_train_data_file.exists(),
		},
		{
			"raw_dataset_file": raw_test_data_file.name,
			"row_count": df_staged_2015_data.shape[0],
			"column_count": df_staged_2015_data.shape[1],
			"file_exists": raw_test_data_file.exists(),
		},
	]
)

df_raw_dataset_creation_summary

,raw_dataset_file,row_count,column_count,file_exists
0,loan_data_2007_2014.csv,466287,111,True
1,loan_data_2015.csv,421095,111,True


## Raw Dataset Construction

The LendingClub data is distributed as several CSV archives covering different time periods. These files are not strict tabular exports. They contain additional rows such as prospectus notices, blank lines, summary totals, and occasional internal headers. Before the data can be treated as a dataset, these structural artifacts must be removed.

### Source normalization

Each archive file is processed line-by-line and written to a staging dataset. Rows that do not belong to the tabular structure are excluded, leaving only loan-level observations with the original column layout preserved. The normalized files are written to the **staging layer**.

### Schema validation

After normalization the column headers of all staged files are compared to verify that the schema is identical across archives. This ensures the files can be combined without structural alignment.

### Raw dataset consolidation

The normalized archives are then consolidated into two canonical datasets.

The project uses a temporal split:

- **2007–2014** → training period  
- **2015** → testing period

The training dataset is created by concatenating the staged files for:

- 2007–2011  
- 2012–2013  
- 2014  

The resulting files are written to the raw data layer:

- `raw_loan_data_2007_2014.csv`  
- `raw_loan_data_2015.csv`

From this point forward all downstream processing operates on these canonical raw datasets rather than on the original archive exports.

## Initial Data Inspection

This step examines the raw dataset.

The objective is to map its structural characteristics before any transformation rules are applied.

The inspection evaluates:

- Column count and schema alignment  
- Datatype assignments and object-encoded numerics  
- Fully null and constant columns  
- Patterns of missingness, including variables where null reflects structural absence  
- Mixed-type inconsistencies  
- Identifier fields and exported artifacts  

No variables are removed and no values are altered at this stage.

Findings from this inspection determine which columns require exclusion, normalization, or explicit encoding during transformation. Every later modification is grounded in observations documented here.

---

In [9]:
# --------------------------------------------------------
# Load raw datasets (train/test) + checkpoint
# --------------------------------------------------------

df_raw_train_data = pd.read_csv(raw_train_data_file, low_memory=False)
df_raw_test_data = pd.read_csv(raw_test_data_file, low_memory=False)

raw_overview_df = pd.DataFrame(
    [
        {
            "dataset": "train_raw",
            "rows": df_raw_train_data.shape[0],
            "cols": df_raw_train_data.shape[1],
            "memory_mb": round(df_raw_train_data.memory_usage(deep=True).sum() / (1024**2), 2),
        },
        {
            "dataset": "test_raw",
            "rows": df_raw_test_data.shape[0],
            "cols": df_raw_test_data.shape[1],
            "memory_mb": round(df_raw_test_data.memory_usage(deep=True).sum() / (1024**2), 2),
        },
    ]
)

display(raw_overview_df)
log(f"[raw] overview: {raw_overview_df.to_dict(orient='records')}")

,dataset,rows,cols,memory_mb
0,train_raw,466287,111,524.51
1,test_raw,421095,111,448.83


In [10]:
# -----------------------------------------
# Raw schema comparison (train vs test)
# -----------------------------------------

combined_schema_raw = da.build_combined_schema(
    df_train=df_raw_train_data,
    df_test=df_raw_test_data,
    stage_name="raw",
    log=log,
)

# Small summary (high signal)
schema_summary_df = pd.DataFrame([{
    "train_rows": df_raw_train_data.shape[0],
    "test_rows": df_raw_test_data.shape[0],
    "train_cols": df_raw_train_data.shape[1],
    "test_cols": df_raw_test_data.shape[1],
    "train_only_count": int((combined_schema_raw["present_in_train"] & ~combined_schema_raw["present_in_test"]).sum()),
    "test_only_count": int((~combined_schema_raw["present_in_train"] & combined_schema_raw["present_in_test"]).sum()),
    "dtype_mismatch_count": int(
        (combined_schema_raw["present_in_train"]
         & combined_schema_raw["present_in_test"]
         & combined_schema_raw["dtype_mismatch"]).sum()
    ),
}])

display(schema_summary_df)
log(f"[raw][schema] notebook_summary: {schema_summary_df.to_dict(orient='records')[0]}")

# Delta-only schema view (columns present in only one split OR dtype mismatch)
schema_deltas_df = combined_schema_raw.loc[
    (combined_schema_raw["present_in_train"] ^ combined_schema_raw["present_in_test"])
    | (
        combined_schema_raw["present_in_train"]
        & combined_schema_raw["present_in_test"]
        & combined_schema_raw["dtype_mismatch"]
    ),
    ["column_name", "train_dtype", "test_dtype", "present_in_train", "present_in_test", "dtype_mismatch"],
].copy()

schema_deltas_df = (
    schema_deltas_df
    .sort_values(["dtype_mismatch", "column_name"], ascending=[False, True])
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(schema_deltas_df)

log(f"[raw][schema] delta_rows={schema_deltas_df.shape[0]}")

,train_rows,test_rows,train_cols,test_cols,train_only_count,test_only_count,dtype_mismatch_count
0,466287,421095,111,111,0,0,38


,column_name,train_dtype,test_dtype,present_in_train,present_in_test,dtype_mismatch
0,acc_now_delinq,float64,int64,True,True,True
1,acc_open_past_24mths,float64,int64,True,True,True
2,avg_cur_bal,float64,int64,True,True,True
3,chargeoff_within_12_mths,float64,int64,True,True,True
4,collections_12_mths_ex_med,float64,int64,True,True,True
5,delinq_2yrs,float64,int64,True,True,True
6,delinq_amnt,float64,int64,True,True,True
7,funded_amnt_inv,float64,int64,True,True,True
8,inq_last_6mths,float64,int64,True,True,True
9,mo_sin_old_rev_tl_op,float64,int64,True,True,True


In [11]:
# ------------------------------------------------------------------
# Feature universe overview (raw train vs raw test)
# For Appendix A
# ------------------------------------------------------------------

inspection_summary_train = da.initial_inspection(df_raw_train_data, PROJECT_LOG_FILE)
inspection_summary_test = da.initial_inspection(df_raw_test_data, PROJECT_LOG_FILE)

feature_summary_train_df = inspection_summary_train["feature_summary"].copy()
feature_summary_test_df = inspection_summary_test["feature_summary"].copy()

train_overview_df = (
    feature_summary_train_df
    .reset_index()
    .rename(columns={
        "index": "column_name",
        "dtype": "train_dtype_inferred",
        "n_unique": "train_n_unique",
        "null_percent": "train_null_percent",
        "is_numeric": "train_is_numeric",
        "is_object": "train_is_object",
        "is_mixed_type": "train_is_mixed_type",
        "is_numeric_object": "train_is_numeric_object",
        "is_fully_null": "train_is_fully_null",
        "is_constant": "train_is_constant",
    })
)

test_overview_df = (
    feature_summary_test_df
    .reset_index()
    .rename(columns={
        "index": "column_name",
        "dtype": "test_dtype_inferred",
        "n_unique": "test_n_unique",
        "null_percent": "test_null_percent",
        "is_numeric": "test_is_numeric",
        "is_object": "test_is_object",
        "is_mixed_type": "test_is_mixed_type",
        "is_numeric_object": "test_is_numeric_object",
        "is_fully_null": "test_is_fully_null",
        "is_constant": "test_is_constant",
    })
)

combined_feature_universe_df = pd.merge(
    train_overview_df,
    test_overview_df,
    on="column_name",
    how="outer",
)

combined_feature_universe_df["max_null_percent"] = combined_feature_universe_df[
    ["train_null_percent", "test_null_percent"]
].max(axis=1)

combined_feature_universe_df["null_gap_test_minus_train"] = (
    combined_feature_universe_df["test_null_percent"] - combined_feature_universe_df["train_null_percent"]
)

combined_feature_universe_df = pd.merge(
    combined_feature_universe_df,
    combined_schema_raw[["column_name", "present_in_train", "present_in_test", "train_dtype", "test_dtype", "dtype_mismatch"]],
    on="column_name",
    how="left",
)

# Severity sorting: mismatches + missingness + structural red flags first
combined_feature_universe_df["severity"] = 0
combined_feature_universe_df.loc[combined_feature_universe_df["dtype_mismatch"] == True, "severity"] += 100
combined_feature_universe_df.loc[
    (combined_feature_universe_df["present_in_train"] ^ combined_feature_universe_df["present_in_test"]) == True,
    "severity"
] += 80
combined_feature_universe_df.loc[
    (combined_feature_universe_df["train_is_fully_null"] == True) | (combined_feature_universe_df["test_is_fully_null"] == True),
    "severity"
] += 60
combined_feature_universe_df.loc[
    (combined_feature_universe_df["train_is_constant"] == True) | (combined_feature_universe_df["test_is_constant"] == True),
    "severity"
] += 40
combined_feature_universe_df.loc[combined_feature_universe_df["max_null_percent"] >= 50, "severity"] += 10

combined_feature_universe_df = (
    combined_feature_universe_df
    .sort_values(["severity", "max_null_percent", "column_name"], ascending=[False, False, True])
    .drop(columns=["severity"])
    .reset_index(drop=True)
)

log(f"[raw][universe] rows={combined_feature_universe_df.shape[0]} cols={combined_feature_universe_df.shape[1]}")


#-------------------------------------------------------------------------
# Actionable insights: columns with dtype mismatches, 
# presence mismatches, high nulls, or constant values
#-------------------------------------------------------------------------

action_df = combined_feature_universe_df.loc[
    (combined_feature_universe_df["dtype_mismatch"] == True)
    | ((combined_feature_universe_df["present_in_train"] ^ combined_feature_universe_df["present_in_test"]) == True)
    | (combined_feature_universe_df["max_null_percent"] >= 50)
    | (combined_feature_universe_df["train_is_constant"] == True)
    | (combined_feature_universe_df["test_is_constant"] == True),
    [
        "column_name",
        "present_in_train",
        "present_in_test",
        "train_dtype",
        "test_dtype",
        "dtype_mismatch",
        "train_null_percent",
        "test_null_percent",
        "max_null_percent",
        "null_gap_test_minus_train",
        "train_is_fully_null",
        "test_is_fully_null",
        "train_is_constant",
        "test_is_constant",
    ],
].copy()

action_df = action_df.sort_values(
    ["dtype_mismatch", "max_null_percent", "column_name"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(action_df)
log(f"[raw][universe] action_rows={action_df.shape[0]}")

,column_name,present_in_train,present_in_test,train_dtype,test_dtype,dtype_mismatch,train_null_percent,test_null_percent,max_null_percent,null_gap_test_minus_train,train_is_fully_null,test_is_fully_null,train_is_constant,test_is_constant
0,verification_status_joint,True,True,float64,str,True,100.00,99.88,100.00,-0.12,True,False,False,False
1,avg_cur_bal,True,True,float64,int64,True,15.07,0.00,15.07,-15.07,False,False,False,False
2,mo_sin_old_rev_tl_op,True,True,float64,int64,True,15.07,0.00,15.07,-15.07,False,False,False,False
3,mo_sin_rcnt_rev_tl_op,True,True,float64,int64,True,15.07,0.00,15.07,-15.07,False,False,False,False
4,mo_sin_rcnt_tl,True,True,float64,int64,True,15.07,0.00,15.07,-15.07,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,mths_since_recent_revol_delinq,True,True,float64,float64,False,70.15,63.97,70.15,-6.18,False,False,False,False
59,next_pymnt_d,True,True,str,str,False,60.07,14.23,60.07,-45.84,False,False,False,False
60,mths_since_last_delinq,True,True,float64,float64,False,53.69,48.44,53.69,-5.25,False,False,False,False
61,application_type,True,True,str,str,False,0.00,0.00,0.00,0.00,False,False,True,False


In [12]:
# -----------------------------------------
# Appendix export: Raw feature universe
# -----------------------------------------

combined_feature_universe_export_df = combined_feature_universe_df.drop(
    columns=["train_dtype_inferred", "test_dtype_inferred"],
    errors="ignore",
).copy()

combined_feature_universe_export_df["train_dtype"] = combined_feature_universe_export_df["train_dtype"].astype("string")
combined_feature_universe_export_df["test_dtype"] = combined_feature_universe_export_df["test_dtype"].astype("string")

combined_feature_universe_export_df.to_csv(appendix_csv_file, index=False, encoding="utf-8")

log(f"[raw][appendix] csv_saved={appendix_csv_file}")

{
    "stage": "appendix_raw_feature_universe",
    "file_saved": True,
    "path": str(appendix_csv_file),
}

{'stage': 'appendix_raw_feature_universe',
 'file_saved': True,
 'path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\audit\\appendix_A_raw_feature_universe.csv'}

In [13]:
# Creating a structural issues report for the raw feature universe
structural_issues_df = da.build_structural_issues_report(
    combined_feature_universe_df=combined_feature_universe_df,
    log=log,
)

# Structural issues summary (high signal)
issue_counts_df = (
    structural_issues_df
    .groupby(["issue"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values(["count", "issue"], ascending=[False, True])
    .reset_index(drop=True)
)

display(issue_counts_df)
log(f"[raw][structural_issues] issue_counts={issue_counts_df.to_dict(orient='records')}")

,issue,count
0,Dtype mismatch,38
1,High Missing (>50%),29
2,100% Null,17
3,Constant,3


In [14]:
# For each identified issue type, we can review example columns to understand the specific problems and potential root causes. This can inform our cleaning and preprocessing strategy in the next steps.
issue_priority = [
    "Present in train only",
    "Present in test only",
    "Dtype mismatch",
    "100% Null",
    "High Missing (>50%)",
    "Constant",
]

examples_df = (
    structural_issues_df
    .assign(issue=pd.Categorical(structural_issues_df["issue"], categories=issue_priority, ordered=True))
    .sort_values(["issue", "column_name", "applies_to"])
    .groupby("issue", as_index=False)
    .head(5)
    .reset_index(drop=True)
)

display(examples_df)
log(f"[raw][structural_issues] example_rows={examples_df.shape[0]}")

,column_name,issue,applies_to
0,acc_now_delinq,Dtype mismatch,both
1,acc_open_past_24mths,Dtype mismatch,both
2,avg_cur_bal,Dtype mismatch,both
3,chargeoff_within_12_mths,Dtype mismatch,both
4,collections_12_mths_ex_med,Dtype mismatch,both
5,all_util,100% Null,train
6,annual_inc_joint,100% Null,train
7,dti_joint,100% Null,train
8,il_util,100% Null,train
9,inq_fi,100% Null,train


In [15]:
# Mapping from issue type to recommended action for cleaning/preprocessing
action_map = {
    "Present in train only": "Drop ingestion artifact / align schema",
    "Present in test only": "Align schema (investigate source) / drop if absent in train",
    "Dtype mismatch": "Coerce to consistent dtype (string/category/numeric)",
    "100% Null": "Drop (non-informative in this cohort)",
    "High Missing (>50%)": "Flag as regime-shift sensitive; likely drop or isolate",
    "Constant": "Drop (non-informative)",
}

decision_queue_df = (
    structural_issues_df
    .assign(recommended_action=structural_issues_df["issue"].map(action_map))
    .drop_duplicates(subset=["column_name", "issue", "applies_to"])
    .sort_values(["issue", "column_name", "applies_to"])
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(decision_queue_df.head(25))
log(f"[raw][structural_issues] decision_queue_rows={decision_queue_df.shape[0]}")

,column_name,issue,applies_to,recommended_action
0,acc_now_delinq,Dtype mismatch,both,Coerce to consistent dtype (string/category/numeric)
1,acc_open_past_24mths,Dtype mismatch,both,Coerce to consistent dtype (string/category/numeric)
2,avg_cur_bal,Dtype mismatch,both,Coerce to consistent dtype (string/category/numeric)
3,chargeoff_within_12_mths,Dtype mismatch,both,Coerce to consistent dtype (string/category/numeric)
4,collections_12_mths_ex_med,Dtype mismatch,both,Coerce to consistent dtype (string/category/numeric)
5,delinq_2yrs,Dtype mismatch,both,Coerce to consistent dtype (string/category/numeric)
6,delinq_amnt,Dtype mismatch,both,Coerce to consistent dtype (string/category/numeric)
7,funded_amnt_inv,Dtype mismatch,both,Coerce to consistent dtype (string/category/numeric)
8,inq_last_6mths,Dtype mismatch,both,Coerce to consistent dtype (string/category/numeric)
9,mo_sin_old_rev_tl_op,Dtype mismatch,both,Coerce to consistent dtype (string/category/numeric)


In [16]:
reports = etla.build_string_alignment_report(
	df_train=df_raw_train_data,
	df_test=df_raw_test_data,
	sample_size=5,
	top_k=30,
	drilldown_max_columns=10,
	drilldown_top_values_per_side=20,
	log=PROJECT_LOG_FILE,
	export_dir=None,
)

display(reports["summary"])
display(reports["top_deltas"])
display(reports["value_differences"])

,string_like_cols_train,string_like_cols_test,string_like_cols_union,string_like_cols_with_differences,top_k_returned
0,24,25,25,18,18


,column_name,dtype_train,unique_including_null_train,unique_non_null_train,null_percent_train,sample_values_train,dtype_test,unique_including_null_test,unique_non_null_test,null_percent_test,sample_values_test,present_in_train,present_in_test,dtype_mismatch,unique_non_null_gap_test_minus_train,null_gap_test_minus_train,max_null_percent,has_difference
0,verification_status_joint,NaN,NaN,NaN,NaN,NaN,str,4,3,99.88,"[Not Verified, Verified, Source Verified]",False,True,False,3.0,NaN,99.88,True
1,next_pymnt_d,str,106.0,105.0,60.07,"[Oct-2016, Sep-2016, Jan-2016, Sep-2013, Feb-2...",str,5,4,14.23,"[Jul-2016, Aug-2016, Jun-2016, May-2016]",True,True,False,-101.0,-45.84,60.07,True
2,desc,str,124436.0,124435.0,72.98,[Borrower added on 12/22/11 > I need to upgrad...,str,35,34,99.99,[We knew that using our credit cards to financ...,True,True,False,-124401.0,27.01,99.99,True
3,emp_title,str,205477.0,205476.0,5.92,"[Ryder, AIR RESOURCES BOARD, University Medica...",str,120813,120812,5.67,"[Road driver, Manager, Director, Paralegal, dr...",True,True,False,-84664.0,-0.25,5.92,True
4,revol_util,str,1270.0,1269.0,0.07,"[83.7%, 9.4%, 98.5%, 21%, 53.9%]",str,1212,1211,0.04,"[65.3%, 50.9%, 9.6%, 42.5%, 35.6%]",True,True,False,-58.0,-0.03,0.07,True
5,title,str,63133.0,63132.0,0.00,"[Computer, bike, real estate business, persone...",str,28,27,0.03,"[Car financing, Other, Debt consolidation, Cre...",True,True,False,-63105.0,0.03,0.03,True
6,last_credit_pull_d,str,112.0,111.0,0.01,"[Sep-2016, Apr-2016, Jan-2016, Dec-2014, Dec-2...",str,20,19,0.00,"[Jun-2016, Apr-2016, May-2016, Mar-2016, Dec-2...",True,True,False,-92.0,-0.01,0.01,True
7,last_pymnt_d,str,107.0,106.0,0.08,"[Jan-2015, Apr-2013, Jun-2014, Sep-2016, May-2...",str,19,18,0.07,"[Jun-2016, Mar-2016, Apr-2016, May-2016, Feb-2...",True,True,False,-88.0,-0.01,0.08,True
8,url,str,466287.0,466287.0,0.00,[https://lendingclub.com/browse/loanDetail.act...,str,421095,421095,0.00,[https://lendingclub.com/browse/loanDetail.act...,True,True,False,-45192.0,0.00,0.00,True
9,int_rate,str,506.0,506.0,0.00,"[10.65%, 15.27%, 15.96%, 13.49%, 12.69%]",str,110,110,0.00,"[7.91%, 14.85%, 9.80%, 5.32%, 11.48%]",True,True,False,-396.0,0.00,0.00,True


,column_name,value,present_in
0,verification_status_joint,Not Verified,test_only
1,verification_status_joint,Source Verified,test_only
2,verification_status_joint,Verified,test_only
3,next_pymnt_d,Apr-2008,train_only
4,next_pymnt_d,Apr-2009,train_only
...,...,...,...
289,int_rate,19.89%,test_only
290,int_rate,24.24%,test_only
291,int_rate,25.09%,test_only
292,int_rate,25.78%,test_only


In [17]:
# -----------------------------------------------------------------------------------
# Raw string alignment checkpoint (train vs test)
# Purpose: detect categorical drift, casing/whitespace issues, and dtype quirks
# before any transformations or column drops.
# -----------------------------------------------------------------------------------

alignment_report = etla.build_string_alignment_report(
    df_train=df_raw_train_data,
    df_test=df_raw_test_data,
    sample_size=5,
    top_k=30,
    drilldown_max_columns=10,
    drilldown_top_values_per_side=20,
    log=PROJECT_LOG_FILE,
    export_dir=audit_dir,
    export_base_name="string_alignment",
    export_tag="raw",
    export_sample_values_as_json=True,
)

log(
    f"[raw][string_alignment] audit_files_created=True "
    f"export_dir={audit_dir}"
)

{
    "stage": "raw_string_alignment",
    "audit_files_created": True,
    "export_dir": str(audit_dir),
}

{'stage': 'raw_string_alignment',
 'audit_files_created': True,
 'export_dir': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\audit'}

# Raw Layer – Structural Governance and Eligibility

The data is provided as two independent source files:

- **2007–2014 vintage (training)**
- **2015 vintage (testing)**

The training vintage defines the admissible feature universe.  
The testing vintage is aligned to it.

> **Note:** Governance decisions below are based on the full audit outputs stored in `artifacts/audit` and the LendingClub Data Dictionary accompanying the dataset. Variable eligibility and lifecycle interpretation follow the documented field definitions.

---

# 1. Initial Data Inspection – Structural Baseline

| Metric | Train (2007–2014) | Test (2015) |
|------|------------------|-------------|
| Rows | **466,287** | **421,095** |
| Columns | **111** | **111** |
| Memory Usage | ~580 MB | ~520 MB |
| Numeric Columns | 87 | 86 |
| Object / String Columns | 24 | 25 |
| Fully Null Columns | **17** | 0 |
| Constant Columns | 2 (`policy_code`, `application_type`) | 1 (`policy_code`) |
| String-like Columns (post dtype scan) | 24 | 25 |
| Columns present only in one split | 0 | 0 |
| Dtype mismatches | 38 | — |

The raw dataset contains **111 variables** across both partitions.

Relative to earlier dataset versions, the wider feature universe primarily reflects **expanded credit bureau reporting fields introduced in later platform vintages**.

The training vintage therefore contains a mixture of:

- stable application variables  
- stable bureau variables  
- later bureau variables with partial historical reporting  
- structural artifacts introduced during export  

Governance rules below formalize which variables remain admissible for modeling.

---

## Fully Null Variables (Training-Defined Constraint)

Seventeen variables are **entirely null in the training data**.

According to the LendingClub Data Dictionary these variables represent later-introduced fields including:

- short-horizon account opening activity windows  
- segmented installment loan utilization measures  
- joint borrower income and verification attributes  
- segmented inquiry counters  

These variables did not exist during the training period. Although a small fraction of observations appear in the testing vintage, they remain **94–99% null**, indicating partial late-period reporting rather than stable availability. A model trained on the 2007–2014 regime cannot estimate variables that did not exist in that regime. All **17 structurally null columns** are therefore removed from both datasets.

---

## Constant Variables and Structural Artifacts

Two columns exhibit zero variance in the training data:

- `policy_code`
- `application_type`

`policy_code` is an internal LendingClub platform designation and contains no variation.

`application_type` indicates whether an application is submitted by an individual borrower or jointly.

In the **2007–2014 training data all applications are individual**.  
Joint applications appear only in the later testing vintage and rely on borrower attributes that are structurally absent from the training data. Including this variable would therefore introduce a borrower structure that never occurs in the training regime. Both variables are removed.

The export artifact `Unnamed: 0` is also removed defensively when present.  
It represents a CSV index column and does not correspond to a documented field in the LendingClub data dictionary.

---

## Credit Timing Variables

The dataset contains several variables measuring the **recency of adverse credit events**:

- `mths_since_last_delinq`
- `mths_since_last_major_derog`
- `mths_since_last_record`
- `mths_since_recent_bc_dlq`
- `mths_since_recent_revol_delinq`

These variables measure the number of months since the borrower last experienced negative credit behavior such as:

- missed payments  
- serious delinquencies  
- public records  

A value of `0` indicates the event occurred within the previous month.  
Higher values indicate events further in the past.

Missing values indicate that **no such event exists in the borrower’s recorded credit history**. High null percentages therefore represent **structural absence** rather than **incomplete reporting**. Dropping these variables would remove meaningful risk signal, while leaving them as `NaN` would treat structural absence as missing data. Handling:

1. Create a binary indicator `has_<variable>`
2. Replace missing values in the original variable with sentinel value **9999**

The sentinel lies well beyond any realistic month count and preserves ordinal interpretation.

---

## Bureau Reporting Expansion

A broader group of bureau variables exhibits **partial historical reporting**. Approximately **15% of observations in the training vintage are missing**, while the same fields are **fully populated in the 2015 testing vintage**. This pattern reflects **expansion of credit bureau reporting coverage over time**, not a transformation error. These variables remain valid predictors because the information they contain exists at application time. To separate **information content** from **reporting availability**, an additional binary indicator is created for each affected variable:

`has_<variable>`

Interpretation:

- `has_* = 0` → field not reported in that vintage  
- `has_* = 1` → value observed  

The original numeric values remain unchanged. This prevents historical reporting gaps from silently introducing a temporal signal into the model.

---

# 2. Submission-Time Boundary

Prediction is defined at the moment the borrower submits the application. The admissible information set therefore consists of:

- borrower-declared application inputs  
- credit bureau data retrieved at submission  

No later platform actions, pricing decisions, funding outcomes, or servicing updates are allowed to enter the training feature space. A variable is eligible only if:

1. It is available at submission  
2. It does not encode underwriting, pricing, funding, or servicing outcomes  
3. It contains usable observations in the training data  

---

# 3. Feature Classification Overview – Application Submission Prediction Point

| Column Group | Category | Action | Rationale |
|------|----------|--------|-----------|
| Application inputs (`loan_amnt`, `term`, `purpose`, `annual_inc`, `dti`<br>`emp_length`, `home_ownership`) | Application input | Keep | Borrower-declared attributes available at submission |
| Credit snapshot (`open_acc`, `total_acc`, `inq_last_6mths`<br>`delinq_2yrs`, `pub_rec`, `collections_12_mths_ex_med`<br>`revol_bal`, `revol_util`, `acc_now_delinq`<br>`tot_cur_bal`, `tot_coll_amt`, `tot_hi_cred_lim`<br>`total_rev_hi_lim`, `total_il_high_credit_limit`) | Credit snapshot | Keep | Credit bureau state at time of application |
| Bureau reporting expansion (`avg_cur_bal`, `mo_sin_old_rev_tl_op`, `mo_sin_rcnt_rev_tl_op`<br>`mo_sin_rcnt_tl`, `num_accts_ever_120_pd`, `num_actv_bc_tl`<br>`num_actv_rev_tl`, `num_bc_tl`, `num_il_tl`<br>`num_op_rev_tl`, `num_rev_tl_bal_gt_0`, `num_tl_30dpd`<br>`num_tl_90g_dpd_24m`, `num_tl_op_past_12m`, `num_rev_accts`) | Credit snapshot | Keep (Indicator Added) | Reporting coverage expanded historically; indicator separates information from reporting availability |
| Credit timing variables (`mths_since_last_delinq`, `mths_since_last_major_derog`, `mths_since_last_record`<br>`mths_since_recent_bc_dlq`, `mths_since_recent_revol_delinq`) | Credit timing | Keep (Transformed) | Structural absence encoded via sentinel and indicator |
| Target (`loan_status`) | Target | Target Only | Defines repayment outcome |
| Platform signals (`grade`, `sub_grade`) | Platform signal | Benchmark Only | LendingClub underwriting classification |
| Pricing outputs (`int_rate`, `installment`) | Pricing output | Benchmark Only | Determined by platform underwriting decision |
| Underwriting outcome (`verification_status`) | Underwriting outcome | Exclude | Determined during verification process |
| Funding decisions (`funded_amnt`, `funded_amnt_inv`) | Funding outcome | Exclude | Reflect investor allocation rather than borrower attributes |
| Lifecycle timestamps (`issue_d`, `initial_list_status`) | Post-submission | Exclude | Determined after application submission |
| Bureau updates (`last_credit_pull_d`) | Post-submission | Exclude | Reflect bureau refresh after origination |
| Servicing variables (`out_prncp`, `out_prncp_inv`, `total_pymnt`, `total_pymnt_inv`<br>`total_rec_prncp`, `total_rec_int`, `total_rec_late_fee`<br>`recoveries`, `collection_recovery_fee`<br>`last_pymnt_d`, `next_pymnt_d`, `pymnt_plan`) | Post-origination | Exclude | Observed only after loan issuance; would introduce outcome leakage |
| Structurally null variables (`annual_inc_joint`, `dti_joint`, `verification_status_joint`<br>`open_acc_6m`, `open_il_6m`, `open_il_12m`, `open_il_24m`<br>`mths_since_rcnt_il`, `total_bal_il`, `il_util`<br>`open_rv_12m`, `open_rv_24m`, `all_util`<br>`inq_fi`, `total_cu_tl`, `inq_last_12m`, `max_bal_bc`) | Vintage dependent | Drop | Absent in training regime |
| Identifiers (`id`, `member_id`, `url`) | Structural | Drop | Non-predictive artifacts |
| Structural variables (`policy_code`, `application_type`) | Structural | Drop | `policy_code` is constant across both datasets; `application_type` has zero variance in the training data and only varies in the testing vintage |
| Free-text (`desc`, `emp_title`, `title`) | Unstructured | Drop | High-cardinality text outside structured modeling scope |
| Geographic proxies (`addr_state`, `zip_code`) | Application input | Drop | Removed to reduce proxy discrimination risk |

---

# 4. String Column Normalization

After structural removal and boundary classification, remaining object-based variables require normalization. Normalization ensures:

- consistent representation across datasets  
- stable typing  
- removal of formatting variation without altering meaning  

Some numeric variables in the raw export are stored as percentage strings rather than numeric values. Two examples are **`int_rate`** (loan interest rate) and **`revol_util`** (revolving credit utilization).  
These appear in the source files in the form `"13.56%"`.

During transformation the percentage symbol is removed and the remaining value is converted to a numeric type so the variables can be treated as numeric quantities during analysis.

| Column | Category | Transformation Action | Training Eligibility | Rationale |
|------|----------|----------------------|---------------------|-----------|
| `term` | Structured categorical | Strip whitespace → extract numeric term (36 / 60) → rename `term_months` → convert to int | Keep | Contractual term |
| `int_rate` | Numeric (percent string) | Remove `%` → convert to float | Benchmark Only | Pricing signal produced by platform underwriting |
| `revol_util` | Numeric (percent string) | Remove `%` → convert to float | Keep | Borrower revolving credit utilization |
| `emp_length` | Ordinal categorical | Map to integer scale 0–10 (`<1` → 0, `10+` → 10) | Keep | Borrower tenure |
| `home_ownership` | Categorical | Normalize casing; map `NONE` → `OTHER` | Keep | Borrower attribute |
| `purpose` | Categorical | Normalize casing | Keep | Loan purpose |
| `grade` | Platform signal | Normalize representation | Benchmark Only | Underwriting output |
| `sub_grade` | Platform signal | Normalize representation | Benchmark Only | Granular signal |
| `verification_status` | Underwriting outcome | Normalize representation | Exclude | Determined after submission |
| `initial_list_status` | Workflow variable | Normalize representation | Exclude | Listing outcome |
| `pymnt_plan` | Servicing indicator | Map `n` → 0, `y` → 1 | Exclude | Post-origination flag |
| `issue_d` | Lifecycle timestamp | Convert to datetime | Exclude | Occurs after submission |
| `last_credit_pull_d` | Lifecycle update | Convert to datetime | Exclude | Bureau refresh |
| `last_pymnt_d` | Servicing timeline | Convert to datetime | Exclude | Post-origination |
| `next_pymnt_d` | Servicing timeline | Convert to datetime | Exclude | Post-origination |
| `loan_status` | Target | No transformation | Target Only | Outcome definition |

---

# 5. Transformation Execution

The governance rules above are implemented in the following sequence.

Execution steps:

- Create a stable internal identifier `row_id`
- Remove structurally ineligible columns
- Align testing data to the training-defined feature universe
- Apply credit timing encoding (`has_*` + sentinel `9999`)
- Create reporting indicators for bureau expansion variables
- Normalize categorical string fields and convert percent-encoded numeric variables
- Convert month-year timestamps to datetime

Outputs are persisted as:

- **`clean`** — governed dataset with benchmark variables retained
- **`feature_base`** — modeling dataset containing only training-eligible predictors and the target

---

In [18]:
# Create row identifier for training data
df_clean_train_data = pp.create_row_identifier(
    df_raw_train_data,
    id_column_name="row_id",
    log=PROJECT_LOG_FILE
)

# Create row identifier for test data
df_clean_test_data = pp.create_row_identifier(
    df_raw_test_data,
    id_column_name="row_id",
    log=PROJECT_LOG_FILE
)

train_unique = df_clean_train_data["row_id"].is_unique
test_unique = df_clean_test_data["row_id"].is_unique

assert train_unique, "row_id not unique in training data"
assert test_unique, "row_id not unique in test data"

log(
    "[row_id_created] "
    f"train_rows={df_clean_train_data.shape[0]} "
    f"test_rows={df_clean_test_data.shape[0]} "
    f"train_unique={train_unique} "
    f"test_unique={test_unique}"
)

{
    "stage": "row_id_created",
    "train_rows": int(df_clean_train_data.shape[0]),
    "test_rows": int(df_clean_test_data.shape[0]),
    "train_unique": train_unique,
    "test_unique": test_unique,
}

{'stage': 'row_id_created',
 'train_rows': 466287,
 'test_rows': 421095,
 'train_unique': True,
 'test_unique': True}

In [19]:
expected_train_rows = df_clean_train_data.shape[0]
expected_test_rows = df_clean_test_data.shape[0]

{
    "stage": "row_count_baseline_recorded",
    "expected_train_rows": int(expected_train_rows),
    "expected_test_rows": int(expected_test_rows),
}

{'stage': 'row_count_baseline_recorded',
 'expected_train_rows': 466287,
 'expected_test_rows': 421095}

In [20]:
# Log sanity checks on the row_id column in both datasets
log_config.log_dataframe_checkpoint(
    df_clean_train_data,
    dataset_name="train",
    checkpoint_name="row_id_created",
    log=log,
    id_column_name="row_id",
)

log_config.log_dataframe_checkpoint(
    df_clean_test_data,
    dataset_name="test",
    checkpoint_name="row_id_created",
    log=log,
    id_column_name="row_id",
)

In [21]:
# Parameters for sentinel encoding of credit history features.
CREDIT_RECENCY_COLUMNS = [
    "mths_since_last_delinq",
    "mths_since_last_record",
    "mths_since_last_major_derog",
    "mths_since_recent_bc_dlq",
    "mths_since_recent_revol_delinq",
]

SENTINEL_VALUE = 9999

# Columns to drop for clean_datasets
STRUCTURAL_DROP_COLUMNS = [
    "annual_inc_joint", "dti_joint", "verification_status_joint", "open_acc_6m", "open_il_6m", "open_il_12m", 
    "open_il_24m","mths_since_rcnt_il", "total_bal_il", "il_util","open_rv_12m", "open_rv_24m", "all_util",
    "inq_fi", "total_cu_tl", "inq_last_12m", "max_bal_bc","id", "member_id","url", "Unnamed: 0", "policy_code", 
    "application_type", "desc", "emp_title", "title", "addr_state", "zip_code"
]

#columns to drop for modeling datasets (after cleaning)
EXCLUDED_COLUMNS = [
    "verification_status", "funded_amnt", "funded_amnt_inv", "issue_d", "initial_list_status",
    "last_credit_pull_d", "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int", 
    "total_rec_late_fee", "recoveries", "collection_recovery_fee", "earliest_cr_line", "out_prncp", 
    "out_prncp_inv", "last_pymnt_d", "next_pymnt_d", "last_pymnt_amnt", "pymnt_plan"
]

BENCHMARK_COLUMNS = ["grade", "sub_grade", "int_rate", "installment"]



In [22]:
# Apply sentinel encoding to the credit recency columns in the training data
df_clean_train_data = pp.apply_credit_sentinel_encoding(
    df_clean_train_data,
    CREDIT_RECENCY_COLUMNS,
    sentinel_value=SENTINEL_VALUE,
    log=PROJECT_LOG_FILE
)

# Apply sentinel encoding to the credit recency columns in the test data
df_clean_test_data = pp.apply_credit_sentinel_encoding(
    df_clean_test_data,
    CREDIT_RECENCY_COLUMNS,
    sentinel_value=SENTINEL_VALUE,
    log=PROJECT_LOG_FILE
)

credit_recency_indicator_columns = [f"has_{column_name}" for column_name in CREDIT_RECENCY_COLUMNS]

train_indicators_present = all(
    column_name in df_clean_train_data.columns
    for column_name in credit_recency_indicator_columns
)
test_indicators_present = all(
    column_name in df_clean_test_data.columns
    for column_name in credit_recency_indicator_columns
)

assert train_indicators_present, "Credit recency indicator columns missing in training data after sentinel encoding."
assert test_indicators_present, "Credit recency indicator columns missing in test data after sentinel encoding."

{
    "stage": "credit_recency_encoded",
    "columns_processed": list(CREDIT_RECENCY_COLUMNS),
    "indicator_columns_created": list(credit_recency_indicator_columns),
    "train_indicators_present": train_indicators_present,
    "test_indicators_present": test_indicators_present,
    "sentinel_value": SENTINEL_VALUE,
}

{'stage': 'credit_recency_encoded',
 'columns_processed': ['mths_since_last_delinq',
  'mths_since_last_record',
  'mths_since_last_major_derog',
  'mths_since_recent_bc_dlq',
  'mths_since_recent_revol_delinq'],
 'indicator_columns_created': ['has_mths_since_last_delinq',
  'has_mths_since_last_record',
  'has_mths_since_last_major_derog',
  'has_mths_since_recent_bc_dlq',
  'has_mths_since_recent_revol_delinq'],
 'train_indicators_present': True,
 'test_indicators_present': True,
 'sentinel_value': 9999}

In [23]:
# Drop columns for cleaning training dataset
df_clean_train_data = fu.drop_columns_with_logging(
    df=df_clean_train_data,
    columns_to_drop=STRUCTURAL_DROP_COLUMNS,
    dataset_name="train_clean",
    log=PROJECT_LOG_FILE,
    errors="ignore",
)

# Drop columns for cleaning test dataset
df_clean_test_data = fu.drop_columns_with_logging(
    df=df_clean_test_data,
    columns_to_drop=STRUCTURAL_DROP_COLUMNS,
    dataset_name="test_clean",
    log=PROJECT_LOG_FILE,
    errors="ignore",
)

{
    "stage": "structural_columns_dropped",
    "columns_requested": len(STRUCTURAL_DROP_COLUMNS),
    "train_columns_after": int(df_clean_train_data.shape[1]),
    "test_columns_after": int(df_clean_test_data.shape[1]),
}

{'stage': 'structural_columns_dropped',
 'columns_requested': 28,
 'train_columns_after': 90,
 'test_columns_after': 90}

In [24]:
# -----------------------------------
# Columns grouped by transformation type
# -----------------------------------

# 1. Text normalization (strip + lowercase + cast to category)
# Used to standardize categorical values and ensure train/test alignment.
categorical_normalization_columns = [
    "home_ownership",
    "purpose",
    "grade",
    "sub_grade",
    "verification_status",
    "initial_list_status",
    "loan_status",
]

categorical_columns_to_cast = [
    "purpose",
    "home_ownership",
    "verification_status",
    "initial_list_status",
    "grade",
    "sub_grade",
    "loan_status",
]

# 2. Deterministic category remapping
# Used to consolidate semantically equivalent labels.
special_categorical_mappings = {
    "home_ownership": {
        "none": "other",
        "any": "other",
    }
}

# 3. Ordinal encoding
# Used for ordered categories with intrinsic rank.
ordinal_encoding_columns = {
    "emp_length": {
        "< 1 year": 0,
        "1 year": 1,
        "2 years": 2,
        "3 years": 3,
        "4 years": 4,
        "5 years": 5,
        "6 years": 6,
        "7 years": 7,
        "8 years": 8,
        "9 years": 9,
        "10+ years": 10
    }
}

# 4. Structured extraction
# Convert contract term to numeric month representation.
term_column = "term"
term_new_name = "term_months"

# 5. Binary encoding
# Convert binary categorical indicators to numeric form.
binary_encoding_columns = {
    "pymnt_plan": {
        "n": 0,
        "y": 1
    }
}

# 6. Month-Year datetime conversion
# Convert month-year strings to datetime format.
month_year_datetime_columns = [
    "issue_d",
    "last_credit_pull_d",
    "last_pymnt_d",
    "next_pymnt_d",
    "earliest_cr_line",
]

# 7. Explicit mapping for listing type
# Replace abbreviated codes with semantic labels.
initial_list_status_mapping = {
    "w": "whole",
    "f": "fractional"
}

# 8. Percent-encoded numeric fields
# Convert percentage strings (e.g. "13.56%") to numeric values.
percent_numeric_columns = [
    "int_rate",
    "revol_util",
]

{
    "stage": "transformation_configuration_defined",
    "categorical_columns": len(categorical_normalization_columns),
    "ordinal_columns": len(ordinal_encoding_columns),
    "binary_columns": len(binary_encoding_columns),
    "datetime_columns": len(month_year_datetime_columns),
    "percent_columns": len(percent_numeric_columns),
}


{'stage': 'transformation_configuration_defined',
 'categorical_columns': 7,
 'ordinal_columns': 1,
 'binary_columns': 1,
 'datetime_columns': 5,
 'percent_columns': 2}

In [25]:
# 1. Normalize text categories (casing/whitespace)
df_clean_train_data = pp.normalize_text_columns(
    df=df_clean_train_data,
    columns_to_normalize=categorical_normalization_columns,
    log=PROJECT_LOG_FILE,
)

df_clean_test_data = pp.normalize_text_columns(
    df=df_clean_test_data,
    columns_to_normalize=categorical_normalization_columns,
    log=PROJECT_LOG_FILE,
)

assert df_clean_train_data.shape[0] > 0
assert df_clean_test_data.shape[0] > 0

{
    "stage": "categorical_text_normalized",
    "columns_processed": list(categorical_normalization_columns),
    "train_columns": int(df_clean_train_data.shape[1]),
    "test_columns": int(df_clean_test_data.shape[1]),
}

{'stage': 'categorical_text_normalized',
 'columns_processed': ['home_ownership',
  'purpose',
  'grade',
  'sub_grade',
  'verification_status',
  'initial_list_status',
  'loan_status'],
 'train_columns': 90,
 'test_columns': 90}

In [26]:
# 2. Home ownership alignment
df_clean_train_data = pp.normalize_home_ownership(
    df_clean_train_data,
    log=PROJECT_LOG_FILE,
)

df_clean_test_data = pp.normalize_home_ownership(
    df_clean_test_data,
    log=PROJECT_LOG_FILE,
)

assert "home_ownership" in df_clean_train_data.columns
assert "home_ownership" in df_clean_test_data.columns

{
    "stage": "home_ownership_normalized",
    "column": "home_ownership",
    "train_rows": int(df_clean_train_data.shape[0]),
    "test_rows": int(df_clean_test_data.shape[0]),
}

{'stage': 'home_ownership_normalized',
 'column': 'home_ownership',
 'train_rows': 466287,
 'test_rows': 421095}

In [27]:
# 3. initial_list_status mapping (w/f -> words)
df_clean_train_data = pp.apply_categorical_mapping(
    df=df_clean_train_data,
    column_name="initial_list_status",
    mapping=initial_list_status_mapping,
    log=PROJECT_LOG_FILE,
    allow_unmapped_values=False,
)

df_clean_test_data = pp.apply_categorical_mapping(
    df=df_clean_test_data,
    column_name="initial_list_status",
    mapping=initial_list_status_mapping,
    log=PROJECT_LOG_FILE,
    allow_unmapped_values=False,
)

assert "initial_list_status" in df_clean_train_data.columns
assert "initial_list_status" in df_clean_test_data.columns

{
    "stage": "initial_list_status_mapped",
    "column": "initial_list_status",
    "train_rows": int(df_clean_train_data.shape[0]),
    "test_rows": int(df_clean_test_data.shape[0]),
}

{'stage': 'initial_list_status_mapped',
 'column': 'initial_list_status',
 'train_rows': 466287,
 'test_rows': 421095}

In [28]:
# 4. pymnt_plan binary
df_clean_train_data = pp.apply_categorical_mapping(
    df=df_clean_train_data,
    column_name="pymnt_plan",
    mapping=binary_encoding_columns["pymnt_plan"],
    log=PROJECT_LOG_FILE,
    allow_unmapped_values=False,
)

df_clean_test_data = pp.apply_categorical_mapping(
    df=df_clean_test_data,
    column_name="pymnt_plan",
    mapping=binary_encoding_columns["pymnt_plan"],
    log=PROJECT_LOG_FILE,
    allow_unmapped_values=False,
)

assert "pymnt_plan" in df_clean_train_data.columns
assert "pymnt_plan" in df_clean_test_data.columns

{
    "stage": "pymnt_plan_binary_encoded",
    "column": "pymnt_plan",
    "train_rows": int(df_clean_train_data.shape[0]),
    "test_rows": int(df_clean_test_data.shape[0]),
}

{'stage': 'pymnt_plan_binary_encoded',
 'column': 'pymnt_plan',
 'train_rows': 466287,
 'test_rows': 421095}

In [29]:
# 5. Term parsing + rename
df_clean_train_data = pp.parse_term_column(
    df=df_clean_train_data,
    term_column=term_column,
    new_column_name=term_new_name,
    log=PROJECT_LOG_FILE,
)

df_clean_test_data = pp.parse_term_column(
    df=df_clean_test_data,
    term_column=term_column,
    new_column_name=term_new_name,
    log=PROJECT_LOG_FILE,
)

assert term_new_name in df_clean_train_data.columns
assert term_new_name in df_clean_test_data.columns

{
    "stage": "term_parsed_to_months",
    "source_column": term_column,
    "new_column": term_new_name,
}

{'stage': 'term_parsed_to_months',
 'source_column': 'term',
 'new_column': 'term_months'}

In [30]:
# 6. emp_length ordinal
df_clean_train_data = pp.transform_emp_length(
    df_clean_train_data,
    log=PROJECT_LOG_FILE,
)

df_clean_test_data = pp.transform_emp_length(
    df_clean_test_data,
    log=PROJECT_LOG_FILE,
)

assert "emp_length" in df_clean_train_data.columns
assert "emp_length" in df_clean_test_data.columns

{
    "stage": "emp_length_ordinal_encoded",
    "column": "emp_length",
    "train_rows": int(df_clean_train_data.shape[0]),
    "test_rows": int(df_clean_test_data.shape[0]),
}

{'stage': 'emp_length_ordinal_encoded',
 'column': 'emp_length',
 'train_rows': 466287,
 'test_rows': 421095}

In [31]:
# 7. Month-year datetime conversions
df_clean_train_data = pp.convert_month_year_columns_to_datetime(
    df=df_clean_train_data,
    column_names=month_year_datetime_columns,
    datetime_formats=["%b-%Y", "%b-%y"],
    log=PROJECT_LOG_FILE,
)

df_clean_test_data = pp.convert_month_year_columns_to_datetime(
    df=df_clean_test_data,
    column_names=month_year_datetime_columns,
    datetime_formats=["%b-%Y", "%b-%y"],
    log=PROJECT_LOG_FILE,
)

for column_name in month_year_datetime_columns:
    assert column_name in df_clean_train_data.columns
    assert column_name in df_clean_test_data.columns

{
    "stage": "month_year_columns_converted_to_datetime",
    "columns_converted": list(month_year_datetime_columns),
    "train_rows": int(df_clean_train_data.shape[0]),
    "test_rows": int(df_clean_test_data.shape[0]),
}

{'stage': 'month_year_columns_converted_to_datetime',
 'columns_converted': ['issue_d',
  'last_credit_pull_d',
  'last_pymnt_d',
  'next_pymnt_d',
  'earliest_cr_line'],
 'train_rows': 466287,
 'test_rows': 421095}

In [32]:
# 8. convert percentage strings to numeric
df_clean_train_data = pp.convert_percent_string_columns_to_numeric(
	df_to_transform=df_clean_train_data,
	percent_columns=percent_numeric_columns,
	log=PROJECT_LOG_FILE,
)

df_clean_test_data = pp.convert_percent_string_columns_to_numeric(
	df_to_transform=df_clean_test_data,
	percent_columns=percent_numeric_columns,
	log=PROJECT_LOG_FILE,
)

for column_name in percent_numeric_columns:
	assert column_name in df_clean_train_data.columns
	assert column_name in df_clean_test_data.columns
 
{  
	"stage": "percent_strings_converted_to_numeric",
	"columns_converted": list(percent_numeric_columns),
	"train_rows": int(df_clean_train_data.shape[0]),
	"test_rows": int(df_clean_test_data.shape[0]),
}

{'stage': 'percent_strings_converted_to_numeric',
 'columns_converted': ['int_rate', 'revol_util'],
 'train_rows': 466287,
 'test_rows': 421095}

In [33]:
# 9. Drop original term and emp_length columns after feature extraction
df_clean_train_data = fu.drop_columns_with_logging(
    df=df_clean_train_data,
    columns_to_drop=["term", "emp_length"],
    dataset_name="train_clean",
    log=PROJECT_LOG_FILE,
    errors="ignore",
)

df_clean_test_data = fu.drop_columns_with_logging(
    df=df_clean_test_data,
    columns_to_drop=["term", "emp_length"],
    dataset_name="test_clean",
    log=PROJECT_LOG_FILE,
    errors="ignore",
)

assert "term" not in df_clean_train_data.columns
assert "term" not in df_clean_test_data.columns
assert "emp_length" not in df_clean_train_data.columns
assert "emp_length" not in df_clean_test_data.columns

{
    "stage": "original_term_and_emp_length_dropped",
    "columns_dropped": ["term", "emp_length"],
    "train_columns_after": int(df_clean_train_data.shape[1]),
    "test_columns_after": int(df_clean_test_data.shape[1]),
}

{'stage': 'original_term_and_emp_length_dropped',
 'columns_dropped': ['term', 'emp_length'],
 'train_columns_after': 90,
 'test_columns_after': 90}

In [34]:
# 10. Standardize categorical dtypes
df_clean_train_data, df_clean_test_data = pp.cast_categorical_columns(
    df_train=df_clean_train_data,
    df_test=df_clean_test_data,
    column_names=categorical_columns_to_cast,
    log=PROJECT_LOG_FILE,
)

for column_name in categorical_columns_to_cast:
    assert column_name in df_clean_train_data.columns
    assert column_name in df_clean_test_data.columns

{
    "stage": "categorical_columns_cast_to_category",
    "columns_cast": list(categorical_columns_to_cast),
    "train_columns": int(df_clean_train_data.shape[1]),
    "test_columns": int(df_clean_test_data.shape[1]),
}

{'stage': 'categorical_columns_cast_to_category',
 'columns_cast': ['purpose',
  'home_ownership',
  'verification_status',
  'initial_list_status',
  'grade',
  'sub_grade',
  'loan_status'],
 'train_columns': 90,
 'test_columns': 90}

In [35]:
# 10. Standardize numerical dtypes
df_clean_train_data, df_clean_test_data = pp.align_numeric_dtypes_between_train_test(
    df_train=df_clean_train_data,
    df_test=df_clean_test_data,
    log=PROJECT_LOG_FILE,
)

{
    "stage": "feature_base_numeric_dtype_alignment",
    "train_columns": df_clean_train_data.shape[1],
    "test_columns": df_clean_test_data.shape[1],
}

{'stage': 'feature_base_numeric_dtype_alignment',
 'train_columns': 90,
 'test_columns': 90}

In [36]:
actual_train_rows = df_clean_train_data.shape[0]
actual_test_rows = df_clean_test_data.shape[0]

if actual_train_rows != expected_train_rows:
    raise ValueError(
        f"Row count drift detected in train dataset. "
        f"expected={expected_train_rows} actual={actual_train_rows}"
    )

if actual_test_rows != expected_test_rows:
    raise ValueError(
        f"Row count drift detected in test dataset. "
        f"expected={expected_test_rows} actual={actual_test_rows}"
    )

{
    "stage": "row_count_invariant_verified",
    "train_rows": int(actual_train_rows),
    "test_rows": int(actual_test_rows),
}

{'stage': 'row_count_invariant_verified',
 'train_rows': 466287,
 'test_rows': 421095}

In [37]:
df_clean_train_text_audit = da.build_residual_text_column_audit(
    df_clean_train_data,
    dataset_split_name="clean_train",
    log=log,
)

df_clean_test_text_audit = da.build_residual_text_column_audit(
    df_clean_test_data,
    dataset_split_name="clean_test",
    log=log,
)

display(df_clean_train_text_audit)
display(df_clean_test_text_audit)

,dataset_split,column_name,dtype,missing_count,missing_rate_percent,unique_count_including_missing,sample_values,suspicious_numeric_like_text
0,clean_train,grade,category,0,0.0,7,b | c | c | c | b,False
1,clean_train,home_ownership,category,0,0.0,4,rent | rent | rent | rent | rent,False
2,clean_train,initial_list_status,category,0,0.0,2,fractional | fractional | fractional | fractio...,False
3,clean_train,loan_status,category,0,0.0,9,fully_paid | charged_off | fully_paid | fully_...,False
4,clean_train,purpose,category,0,0.0,14,credit_card | car | small_business | other | o...,False
5,clean_train,sub_grade,category,0,0.0,35,b2 | c4 | c5 | c1 | b5,False
6,clean_train,verification_status,category,0,0.0,3,verified | source_verified | not_verified | so...,False


,dataset_split,column_name,dtype,missing_count,missing_rate_percent,unique_count_including_missing,sample_values,suspicious_numeric_like_text
0,clean_test,grade,category,0,0.0,7,a | c | b | a | b,False
1,clean_test,home_ownership,category,0,0.0,4,mortgage | mortgage | rent | mortgage | rent,False
2,clean_test,initial_list_status,category,0,0.0,2,whole | whole | whole | whole | whole,False
3,clean_test,loan_status,category,0,0.0,7,current | current | current | current | current,False
4,clean_test,purpose,category,0,0.0,14,car | other | debt_consolidation | credit_card...,False
5,clean_test,sub_grade,category,0,0.0,35,a5 | c5 | b3 | a1 | b5,False
6,clean_test,verification_status,category,0,0.0,3,not_verified | not_verified | not_verified | n...,False


In [38]:
# -----------------------------------------------------------------------------------
# Clean string alignment checkpoint (train vs test)
# Purpose: detect categorical drift and unexpected value changes
# after transformations and column drops.
# -----------------------------------------------------------------------------------

string_alignment_report = etla.build_string_alignment_report(
    df_train=df_clean_train_data,
    df_test=df_clean_test_data,
    sample_size=5,
    top_k=30,
    drilldown_max_columns=10,
    drilldown_top_values_per_side=20,
    log=PROJECT_LOG_FILE,
    export_dir=audit_dir,
    export_base_name="string_alignment",
    export_tag="clean",
    export_sample_values_as_json=True,
)

log(
    f"[clean][string_alignment] audit_files_created=True "
    f"export_dir={audit_dir}"
)

{
    "stage": "clean_string_alignment",
    "audit_files_created": True,
    "export_dir": str(audit_dir),
}

{'stage': 'clean_string_alignment',
 'audit_files_created': True,
 'export_dir': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\audit'}

In [39]:
# -----------------------------------------------------------------------------------
# Feature base numerical alignment checkpoint (train vs test)
# Purpose: detect numeric drift after feature selection
# -----------------------------------------------------------------------------------

numerical_alignment_report = etla.build_numerical_alignment_report(
    df_train=df_clean_train_data,
    df_test=df_clean_test_data,
    top_k=30,
    log=PROJECT_LOG_FILE,
    export_dir=audit_dir,
    export_base_name="numerical_alignment",
    export_tag="clean",
)

log(
    f"[feature_base][numerical_alignment] audit_files_created=True "
    f"export_dir={audit_dir}"
)

{
    "stage": "feature_base_numerical_alignment",
    "audit_files_created": True,
    "export_dir": str(audit_dir),
}

{'stage': 'feature_base_numerical_alignment',
 'audit_files_created': True,
 'export_dir': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\audit'}

In [42]:
# ------------------------------------------------------------------------------
# Create feature_base datasets by removing excluded + benchmark columns
# ------------------------------------------------------------------------------

df_feature_base_train_data = fu.drop_columns_with_logging(
    df=df_clean_train_data,
    columns_to_drop=EXCLUDED_COLUMNS + BENCHMARK_COLUMNS,
    dataset_name="train_feature_base",
    log=PROJECT_LOG_FILE,
    errors="raise",
)

df_feature_base_test_data = fu.drop_columns_with_logging(
    df=df_clean_test_data,
    columns_to_drop=EXCLUDED_COLUMNS + BENCHMARK_COLUMNS,
    dataset_name="test_feature_base",
    log=PROJECT_LOG_FILE,
    errors="raise",
)

for column_name in EXCLUDED_COLUMNS + BENCHMARK_COLUMNS:
    assert column_name not in df_feature_base_train_data.columns
    assert column_name not in df_feature_base_test_data.columns

{
    "stage": "feature_base_created",
    "dropped_columns": len(EXCLUDED_COLUMNS + BENCHMARK_COLUMNS),
    "train_columns": int(df_feature_base_train_data.shape[1]),
    "test_columns": int(df_feature_base_test_data.shape[1]),
}

{'stage': 'feature_base_created',
 'dropped_columns': 24,
 'train_columns': 66,
 'test_columns': 66}

In [43]:
df_feature_base_train_text_audit = da.build_residual_text_column_audit(
    df_feature_base_train_data,
    dataset_split_name="feature_base_train",
    log=log,
)

df_feature_base_test_text_audit = da.build_residual_text_column_audit(
    df_feature_base_test_data,
    dataset_split_name="feature_base_test",
    log=log,
)

display(df_feature_base_train_text_audit)
display(df_feature_base_test_text_audit)

,dataset_split,column_name,dtype,missing_count,missing_rate_percent,unique_count_including_missing,sample_values,suspicious_numeric_like_text
0,feature_base_train,home_ownership,category,0,0.0,4,rent | rent | rent | rent | rent,False
1,feature_base_train,loan_status,category,0,0.0,9,fully_paid | charged_off | fully_paid | fully_...,False
2,feature_base_train,purpose,category,0,0.0,14,credit_card | car | small_business | other | o...,False


,dataset_split,column_name,dtype,missing_count,missing_rate_percent,unique_count_including_missing,sample_values,suspicious_numeric_like_text
0,feature_base_test,home_ownership,category,0,0.0,4,mortgage | mortgage | rent | mortgage | rent,False
1,feature_base_test,loan_status,category,0,0.0,7,current | current | current | current | current,False
2,feature_base_test,purpose,category,0,0.0,14,car | other | debt_consolidation | credit_card...,False


In [44]:
# -----------------------------------------------------------------------------------
# Feature base string alignment checkpoint (train vs test)
# Purpose: detect categorical drift after feature selection
# -----------------------------------------------------------------------------------

string_alignment_report = etla.build_string_alignment_report(
    df_train=df_feature_base_train_data,
    df_test=df_feature_base_test_data,
    sample_size=5,
    top_k=30,
    drilldown_max_columns=10,
    drilldown_top_values_per_side=20,
    log=PROJECT_LOG_FILE,
    export_dir=audit_dir,
    export_base_name="string_alignment",
    export_tag="feature_base",
    export_sample_values_as_json=True,
)

log(
    f"[feature_base][string_alignment] audit_files_created=True "
    f"export_dir={audit_dir}"
)

{
    "stage": "feature_base_string_alignment",
    "audit_files_created": True,
    "export_dir": str(audit_dir),
}

{'stage': 'feature_base_string_alignment',
 'audit_files_created': True,
 'export_dir': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\audit'}

In [45]:
# -----------------------------------------------------------------------------------
# Feature base numerical alignment checkpoint (train vs test)
# Purpose: detect numeric drift after feature selection
# -----------------------------------------------------------------------------------

numerical_alignment_report = etla.build_numerical_alignment_report(
    df_train=df_feature_base_train_data,
    df_test=df_feature_base_test_data,
    top_k=30,
    log=PROJECT_LOG_FILE,
    export_dir=audit_dir,
    export_base_name="numerical_alignment",
    export_tag="feature_base",
)

log(
    f"[feature_base][numerical_alignment] audit_files_created=True "
    f"export_dir={audit_dir}"
)

{
    "stage": "feature_base_numerical_alignment",
    "audit_files_created": True,
    "export_dir": str(audit_dir),
}

{'stage': 'feature_base_numerical_alignment',
 'audit_files_created': True,
 'export_dir': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\audit'}

In [46]:
# --------------------------------------------------
# Feature_base schema alignment check
# --------------------------------------------------

train_columns = df_feature_base_train_data.columns.tolist()
test_columns = df_feature_base_test_data.columns.tolist()

columns_only_in_train = sorted(set(train_columns) - set(test_columns))
columns_only_in_test = sorted(set(test_columns) - set(train_columns))

schema_match = not columns_only_in_train and not columns_only_in_test

log(
    "[feature_base][schema_check] "
    f"schema_match={schema_match} "
    f"train_columns={len(train_columns)} "
    f"test_columns={len(test_columns)} "
    f"only_in_train={len(columns_only_in_train)} "
    f"only_in_test={len(columns_only_in_test)}"
)

assert schema_match, (
    "Train/test feature_base columns are not identical.\n"
    f"Only in train: {columns_only_in_train}\n"
    f"Only in test: {columns_only_in_test}"
)

{
    "stage": "feature_base_schema_alignment",
    "schema_match": schema_match,
    "column_count": len(train_columns),
}

{'stage': 'feature_base_schema_alignment',
 'schema_match': True,
 'column_count': 66}

In [47]:
# --------------------------------------------------
# Feature_base dtype alignment check
# --------------------------------------------------

dtype_mismatches = []

for column_name in train_columns:
    train_dtype = df_feature_base_train_data[column_name].dtype
    test_dtype = df_feature_base_test_data[column_name].dtype

    if train_dtype != test_dtype:
        dtype_mismatches.append(
            (column_name, str(train_dtype), str(test_dtype))
        )

assert not dtype_mismatches, (
    "Dtype mismatches detected between train and test feature_base datasets.\n"
    f"First mismatches: {dtype_mismatches[:20]}"
)

log(
    "[feature_base][dtype_check] "
    f"mismatches={len(dtype_mismatches)} "
    f"columns_checked={len(train_columns)}"
)

{
    "stage": "feature_base_dtype_alignment",
    "dtype_mismatches": len(dtype_mismatches),
    "columns_checked": len(train_columns),
}

{'stage': 'feature_base_dtype_alignment',
 'dtype_mismatches': 0,
 'columns_checked': 66}

In [48]:
# --------------------------------------------------
# Save clean and feature_base datasets (parquet)
# --------------------------------------------------

log = log_config.get_logger(PROJECT_LOG_FILE)

log(
    "[save_datasets] "
    f"train_clean={df_clean_train_data.shape} | "
    f"test_clean={df_clean_test_data.shape} | "
    f"train_feature_base={df_feature_base_train_data.shape} | "
    f"test_feature_base={df_feature_base_test_data.shape}"
)

log(f"[save_datasets] dataset=train_clean path={clean_train_data_file}")
df_clean_train_data.to_parquet(clean_train_data_file, index=False)

log(f"[save_datasets] dataset=test_clean path={clean_test_data_file}")
df_clean_test_data.to_parquet(clean_test_data_file, index=False)

log(f"[save_datasets] dataset=train_feature_base path={feature_base_train_data_file}")
df_feature_base_train_data.to_parquet(feature_base_train_data_file, index=False)

log(f"[save_datasets] dataset=test_feature_base path={feature_base_test_data_file}")
df_feature_base_test_data.to_parquet(feature_base_test_data_file, index=False)

{
    "stage": "datasets_saved",
    "train_clean_rows": int(df_clean_train_data.shape[0]),
    "test_clean_rows": int(df_clean_test_data.shape[0]),
    "train_feature_base_rows": int(df_feature_base_train_data.shape[0]),
    "test_feature_base_rows": int(df_feature_base_test_data.shape[0]),
}

{'stage': 'datasets_saved',
 'train_clean_rows': 466287,
 'test_clean_rows': 421095,
 'train_feature_base_rows': 466287,
 'test_feature_base_rows': 421095}

In [49]:
# Data layer summary table
shape_summary = pd.DataFrame(
    [
        ("raw_train", *df_raw_train_data.shape),
        ("raw_test", *df_raw_test_data.shape),
        ("clean_train", *df_clean_train_data.shape),
        ("clean_test", *df_clean_test_data.shape),
        ("feature_base_train", *df_feature_base_train_data.shape),
        ("feature_base_test", *df_feature_base_test_data.shape),
    ],
    columns=["dataset", "rows", "columns"],
)

display(shape_summary)

,dataset,rows,columns
0,raw_train,466287,111
1,raw_test,421095,111
2,clean_train,466287,90
3,clean_test,421095,90
4,feature_base_train,466287,66
5,feature_base_test,421095,66


# Notebook Conclusion – Data Cleaning and Feature Base Construction

This notebook establishes the transformation layer required for the **Loans at Risk: Capturing Default** project.

The governing constraint is that prediction must rely **only on information available at the moment a borrower submits a loan application**. All transformation decisions follow from that boundary. The raw LendingClub exports contain **111 variables** across both dataset partitions. Through structural governance and transformation, these are reduced to two controlled dataset layers:

- **clean** – a normalized dataset in which variables retain their economic meaning but are stored using consistent datatypes
- **feature_base** – the modeling dataset containing only variables admissible under the submission-time constraint

Key transformations performed in this notebook include:

- removal of structurally ineligible or non-predictive variables  
- encoding of credit timing variables to distinguish structural absence from observed events  
- separation of bureau information from historical reporting availability  
- normalization of categorical string fields  
- conversion of percent-encoded numeric variables to numeric form  
- conversion of month-year timestamps to datetime representations  

The resulting datasets maintain identical schemas across the training and testing partitions and preserve row counts throughout transformation. The **clean dataset** contains **90 variables**, while the **feature_base dataset** contains **66 variables** that satisfy the submission-time eligibility rules.

The transformation layer therefore removes three structural risks before modeling begins: information leakage from post-submission variables, misinterpretation of structurally absent credit events as missing data, and spurious temporal signals introduced by historical reporting expansion. This feature space forms the controlled input layer for the subsequent exploratory analysis and modeling stages of the project.